In [1]:
from pyi18next.utility import get_plural_func
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from adaptation.graph_builder import LocalizationGraphBuilder
from adaptation.misc import traverse_namespaces, NameAnonymizer
import os


In [2]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} bert={'es': 'dccuchile/bert-base-spanish-wwm-cased'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/spanish_word2vec/word2vec.bin'} spacy={'es': 'es_core_news_sm'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/lstm_mean_cosine_noaug'} allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000


In [3]:
base_dir = "./adaptation/localization/final"


In [4]:
namespaces = traverse_namespaces(base_dir, languages)
print(namespaces)


['computer/socialMediaScreen', 'scene1/scene1Break', 'scene1/scene1Lunch2', 'scene4/scene4Garage', 'scene5/scene5Bedroom', 'menus/creditsScene', 'scene6/routeA/scene6PortalRouteA', 'scene6/routeB/scene6PoliceStationRouteB', 'scene5/scene5Livingroom', 'scene6/scene6Bedroom', 'dialogManager', 'scene7/scene7Bedroom', 'scene4/scene4Backyard', 'scene6/routeA/scene6LunchRouteA', 'scene1/scene1Bedroom1', 'scene6/routeB/scene6LunchRouteB', 'transitions', 'menus/titleScene', 'scene6/routeA/scene6BedroomRouteA1', 'scene2/scene2Break', 'scene3/scene3Break', 'scene6/routeA/scene6BedroomRouteA2', 'scene6/routeB/scene6BedroomRouteB', 'deviceInfo', 'scene1/scene1Classroom', 'generalDialogs', 'scene6/routeA/scene6EndingRouteA', 'scene1/scene1Lunch1', 'computer/captions', 'scene6/routeB/scene6EndingRouteB', 'computer/usernames', 'computer/loginScreen', 'scene1/scene1Bedroom2', 'scene3/scene3Bedroom', 'scene6/scene6Livingroom', 'scene4/scene4Frontyard', 'names', 'menus/loginScene', 'scene2/scene2Bedroom

In [5]:
backend = Backend(name_mapping=lambda lng, ns: f"{base_dir}/{lng}/{ns}.json")

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [6]:
# rules = "one: n is 1; other:"
rules = {
	"one": "n is 1",
	"other": ""
}

plural_func = get_plural_func(rules)

print(plural_func(1))
print(plural_func(3))


one
other


In [7]:
data_dir = "./adaptation/data"
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [8]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)

# print(name_anonymizer.whitelist)


In [9]:
base_dir = "./faiss_data"

model_registry = ModelRegistry(languages)
model_registry.build_tranformer("sbert")
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=base_dir
)
model_types = model_registry.active_model_types()


2026-06-11 04:28:21.718 | DEBUG    | services.model_registry:_create_loader:60 - Registering sbert loader for 'es'
2026-06-11 04:28:21.720 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-06-11 04:28:24.092 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'


In [10]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir="./adaptation/localization/structure",
)

builder.run()


scene1/scene1Bedroom1


2026-06-11 04:28:24.703 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 80 vectors
2026-06-11 04:28:24.766 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 48 vectors
2026-06-11 04:28:24.781 | DEBUG    | services.node_engine:save_node:44 - Saving FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer1_choices
2026-06-11 04:28:24.796 | DEBUG    | services.node_engine:save_node:44 - Saving FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2


scene1/scene1Classroom
Total visited nodes: 667


In [11]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=base_dir
)
test_engine = multilingual.get_node_engine("es", "sbert")

print(test_engine.retrievers)

test_engine.load_all()

print(test_engine.retrievers)


2026-06-11 04:28:24.818 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer1_choices
2026-06-11 04:28:24.821 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-06-11 04:28:24.823 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks2
2026-06-11 04:28:24.825 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.


{}
{'scene1Bedroom1_computer1_choices': <controllers.retrievers.faiss.FaissRetriever object at 0x0000023DE68AA480>, 'scene1Classroom_part2_thanks2': <controllers.retrievers.faiss.FaissRetriever object at 0x0000023DE68ABA40>}


In [12]:
retriever = test_engine.get_retriever("scene1Classroom_part2_thanks2")

retriever.search("Hola", 3)


(array([42, 35, 17], dtype=int32),
 array([0.99999994, 0.5333565 , 0.5254723 ], dtype=float32),
 array(['Hola', 'Buenas! Soy [UNK] encantado.', 'Holaaa, soy [UNK] que ta'],
       dtype=object))